# Reducción de Dimensionalidad y Clasificación
## PCA + K-Means + SVM sobre MNIST Digit Recognizer

**Estudiante:** José Carlos Torres Donaire — 20211920141
**Materia:** Inteligencia Artificial

**Objetivo:** Aplicar PCA para reducir dimensionalidad, K-Means para agrupar observaciones, y SVM para clasificar dígitos manuscritos del dataset MNIST.

## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline
print('Librerías importadas correctamente')

Librerías importadas correctamente


## 2. Carga de datos

In [2]:
import os

DATA_PATH = 'data/train.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'No se encuentra {DATA_PATH}. '
        'Descarga train.csv desde https://www.kaggle.com/competitions/digit-recognizer/data '
        'y colocalo en la carpeta data/'
    )

df = pd.read_csv(DATA_PATH)
print(f'Dimensiones del dataset: {df.shape}')
print(f'Columnas: {df.columns.tolist()[:10]}... (total {len(df.columns)} columnas)')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

### Distribución de clases

In [ ]:
plt.figure(figsize=(10, 4))
df['label'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.xlabel('Dígito')
plt.ylabel('Frecuencia')
plt.title('Distribución de clases en el dataset')
plt.show()

print(f'\nTotal muestras: {len(df):,}')
print(f'Clases: {sorted(df["label"].unique())}')

### Visualización de dígitos de ejemplo

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for digit in range(10):
    idx = df[df['label'] == digit].index[0]
    img = df.drop(columns=['label']).iloc[idx].values.reshape(28, 28)
    for row_idx in range(2):
        axes[row_idx][digit].imshow(img, cmap='gray')
        axes[row_idx][digit].axis('off')
        if row_idx == 0:
            axes[row_idx][digit].set_title(str(digit))
fig.suptitle('Ejemplos de dígitos manuscritos', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 3. Preprocesamiento

Se estandarizan los valores de píxeles (media 0, varianza 1) y se divide en entrenamiento (80%) y prueba (20%). Se usa una muestra de 10,000 registros para acelerar el proceso.

In [ ]:
SAMPLE_SIZE = 10000
TEST_SIZE = 0.2
RANDOM_STATE = 42

X = df.drop(columns=['label']).values.astype(np.float32)
y = df['label'].values

rng = np.random.RandomState(RANDOM_STATE)
idx = rng.choice(len(X), SAMPLE_SIZE, replace=False)
X, y = X[idx], y[idx]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test:  {y_test.shape}')

## 4. PCA - Reducción de Dimensionalidad

Aplicamos PCA con diferentes números de componentes para analizar la varianza explicada.

In [ ]:
N_COMPONENTS = 50

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f'Varianza explicada por los primeros {N_COMPONENTS} componentes:')
for i in range(min(10, N_COMPONENTS)):
    print(f'  PC{i+1}: {explained_variance[i]*100:.2f}% (acumulado: {cumulative_variance[i]*100:.2f}%)')
print(f'\nVarianza total explicada (primeros {N_COMPONENTS} componentes): {cumulative_variance[-1]*100:.2f}%')

### Visualización de varianza explicada

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de varianza individual
ax1.bar(range(1, N_COMPONENTS + 1), explained_variance * 100, alpha=0.7)
ax1.set_xlabel('Componente principal')
ax1.set_ylabel('Varianza explicada (%)')
ax1.set_title('Varianza individual por componente')
ax1.grid(alpha=0.3)

# Gráfico de varianza acumulada
ax2.plot(range(1, N_COMPONENTS + 1), cumulative_variance * 100, 'r-o', markersize=3)
ax2.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='90%')
ax2.axhline(y=95, color='orange', linestyle='--', alpha=0.5, label='95%')
ax2.axhline(y=99, color='red', linestyle='--', alpha=0.5, label='99%')
ax2.set_xlabel('Número de componentes')
ax2.set_ylabel('Varianza acumulada (%)')
ax2.set_title('Varianza acumulada')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Encontrar cuántos componentes se necesitan para ciertos umbrales
for threshold in [0.90, 0.95, 0.99]:
    n = np.argmax(cumulative_variance >= threshold) + 1
    print(f'Para {threshold*100:.0f}% de varianza: {n} componentes')

### Proyección 2D de los datos

In [ ]:
# PCA con 2 componentes para visualización
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca_2d.fit_transform(X_train)

df_2d = pd.DataFrame({
    'PC1': X_train_2d[:, 0],
    'PC2': X_train_2d[:, 1],
    'Dígito': y_train.astype(str)
})

plt.figure(figsize=(12, 8))
for digit in range(10):
    subset = df_2d[df_2d['Dígito'] == str(digit)]
    plt.scatter(subset['PC1'], subset['PC2'], label=str(digit), alpha=0.5, s=10)
plt.xlabel('Primer Componente Principal')
plt.ylabel('Segundo Componente Principal')
plt.title('Proyección PCA en 2 dimensiones')
plt.legend(title='Dígito', bbox_to_anchor=(1.05, 1))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Varianza explicada con 2 componentes: {pca_2d.explained_variance_ratio_.sum()*100:.2f}%')

## 5. K-Means - Agrupamiento no supervisado

Aplicamos K-Means sobre los datos transformados por PCA para visualizar la formación de clústeres.

In [ ]:
N_CLUSTERS = 10  # mismo número que clases (0-9)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_train_pca[:, :10])

print(f'Inercia del modelo: {kmeans.inertia_:,.0f}')
print(f'Centroides: {kmeans.cluster_centers_.shape}')

### Visualización de clústeres vs etiquetas reales

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

X_2d_viz = X_train_pca[:, :2]

# Clústeres K-Means
scatter_left = ax_left.scatter(
    X_2d_viz[:, 0], X_2d_viz[:, 1], c=cluster_labels,
    cmap='tab10', alpha=0.6, s=10
)
ax_left.set_xlabel('PC1')
ax_left.set_ylabel('PC2')
ax_left.set_title('Clústeres formados por K-Means')
plt.colorbar(scatter_left, ax=ax_left, label='Cluster')

# Etiquetas reales
scatter_right = ax_right.scatter(
    X_2d_viz[:, 0], X_2d_viz[:, 1], c=y_train,
    cmap='tab10', alpha=0.6, s=10
)
ax_right.set_xlabel('PC1')
ax_right.set_ylabel('PC2')
ax_right.set_title('Etiquetas reales (para comparación)')
plt.colorbar(scatter_right, ax=ax_right, label='Dígito real')

plt.tight_layout()
plt.show()

### Composición de cada clúster

In [ ]:
cross_tab = pd.crosstab(cluster_labels, y_train, margins=True)
cross_tab

# Precisión del agrupamiento (asignando cada cluster al dígito mayoritario)
cluster_map = {}
for cluster in range(N_CLUSTERS):
    mask = cluster_labels == cluster
    dominant_digit = pd.Series(y_train[mask]).mode()[0]
    accuracy = (y_train[mask] == dominant_digit).mean()
    cluster_map[cluster] = {'dígito_asignado': dominant_digit, 'precisión': f'{accuracy*100:.1f}%'}

pd.DataFrame(cluster_map).T

## 6. SVM - Clasificación supervisada

Entrenamos un SVM sobre los datos reducidos por PCA para clasificar los dígitos.

In [ ]:
# Usamos los primeros 10 componentes PCA para SVM
N_COMPONENTS_SVM = 10

pca_svm = PCA(n_components=N_COMPONENTS_SVM, random_state=RANDOM_STATE)
X_train_svm = pca_svm.fit_transform(X_train)
X_test_svm = pca_svm.transform(X_test)

print(f'Entrenamiento: {X_train_svm.shape}')
print(f'Prueba: {X_test_svm.shape}')

In [ ]:
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
svm_model.fit(X_train_svm, y_train)
print('SVM entrenado correctamente')

In [ ]:
y_pred = svm_model.predict(X_test_svm)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy del modelo: {accuracy*100:.2f}%')

### Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de Confusión - SVM con PCA')
plt.show()

### Reporte de Clasificación

In [ ]:
report = classification_report(y_test, y_pred, output_dict=True)
pd.DataFrame(report).transpose().round(3)

### Precisión por dígito

In [ ]:
print('Precisión por dígito:')
for d in range(10):
    mask = y_test == d
    acc = (y_pred[mask] == d).sum() / mask.sum() * 100
    print(f'  Dígito {d}: {acc:.1f}% ({mask.sum():.0f} muestras)')

### Predicción sobre ejemplos individuales

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = i * 50
    img = X_test[idx].reshape(28, 28)
    pred = svm_model.predict(X_test_svm[idx].reshape(1, -1))[0]
    true = y_test[idx]
    color = 'green' if pred == true else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Real: {true} | Pred: {pred}', color=color, fontsize=10)
    ax.axis('off')
plt.suptitle('Predicciones del modelo SVM + PCA', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Comparación: PCA + SVM vs SVM sin reducción

Comparamos el rendimiento y tiempo de entrenamiento con y sin PCA.

In [ ]:
import time

# SVM con PCA (10 componentes)
start = time.time()
svm_pca = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
svm_pca.fit(X_train_svm, y_train)
time_pca = time.time() - start
acc_pca = accuracy_score(y_test, svm_pca.predict(X_test_svm))

# SVM sin PCA (solo con 1000 muestras para que sea factible)
start = time.time()
svm_full = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
svm_full.fit(X_train[:1000], y_train[:1000])
time_full = time.time() - start
acc_full = accuracy_score(y_test, svm_full.predict(X_test))

print('Comparación:')
print(f'  {'':20s} {'PCA+SVM':>12s} {'SVM completo':>14s}')
print(f'  {'Dimensionalidad':20s} {X_train_svm.shape[1]:>6d}          {X_train.shape[1]:>6d}')
print(f'  {'Tiempo entrenamiento':20s} {time_pca:>6.2f}s          {time_full:>6.2f}s')
print(f'  {'Accuracy':20s} {acc_pca*100:>6.2f}%          {acc_full*100:>6.2f}%')

## 8. Análisis de resultados

**Efecto de la reducción de dimensionalidad con PCA:**

- PCA reduce las 784 dimensiones originales a un número manejable de componentes (ej: 10-50).
- Con solo 10 componentes se captura ~50% de la varianza y se obtiene una accuracy competitiva.
- El entrenamiento de SVM es significativamente más rápido con los datos reducidos.
- La proyección 2D permite visualizar la separabilidad de las clases.
- K-Means sobre datos PCA muestra agrupamientos que se correlacionan con los dígitos reales.

**Conclusión:** PCA + SVM ofrece un balance óptimo entre precisión y eficiencia computacional para clasificación de imágenes.

## 9. Código de la aplicación Streamlit

El archivo `app.py` contiene la aplicación interactiva para desplegar en Streamlit Community Cloud.
Ejecutar con: `streamlit run app.py`